# 🧪 Lab 08: Schema Design Patterns

**Topics:** Bucket, Computed, Polymorphic, Outlier, Subset

---

In [ ]:
from pymongo import MongoClient, ReadPreference
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern
from pymongo.errors import ConnectionFailure
from bson import ObjectId
from datetime import datetime, timedelta, timezone
import pprint

USE_ATLAS = False
ATLAS_URI  = "mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority"
DOCKER_URI = "mongodb://127.0.0.1:27017/?directConnection=true"

uri = ATLAS_URI if USE_ATLAS else DOCKER_URI
client = MongoClient(uri, serverSelectionTimeoutMS=5000)
try:
    client.admin.command("ping")
    print("✅ Connected to", "Atlas" if USE_ATLAS else "Docker MongoDB")
except ConnectionFailure as e:
    print("❌ Connection failed:", e)

db = client["mongo_labs"]
pp = pprint.PrettyPrinter(indent=2)

## Bucket Pattern — Time-series

In [ ]:
db.sensor_buckets.drop()
db.sensor_buckets.create_index([("sensorId",1),("hour",1)],unique=True)
sensor_id, hour = "sensor-001", "2026-04-10T10:00"
readings = [{"ts":datetime(2026,4,10,10,i+1),"temp":22.0+i*0.1} for i in range(5)]
for r in readings:
    db.sensor_buckets.update_one({"sensorId":sensor_id,"hour":hour},{"$push":{"readings":r},"$inc":{"count":1},"$min":{"minTemp":r["temp"]},"$max":{"maxTemp":r["temp"]},"$setOnInsert":{"sensorId":sensor_id,"hour":hour,"createdAt":datetime.now(timezone.utc)}},upsert=True)
bucket = db.sensor_buckets.find_one({"sensorId":sensor_id})
print(f"Bucket: {bucket['count']} readings, temp range {bucket['minTemp']:.1f}–{bucket['maxTemp']:.1f}°C")

## Computed Pattern — Pre-computed stats

In [ ]:
db.products_computed.drop()
db.products_computed.insert_one({"_id":"prod-001","name":"Keyboard","ratingStats":{"total":450,"count":100,"avg":4.5}})
new_rating = 5
db.products_computed.update_one({"_id":"prod-001"},{"$inc":{"ratingStats.total":new_rating,"ratingStats.count":1}})
prod = db.products_computed.find_one({"_id":"prod-001"})
new_avg = prod["ratingStats"]["total"] / prod["ratingStats"]["count"]
db.products_computed.update_one({"_id":"prod-001"},{"$set":{"ratingStats.avg":new_avg}})
print(f"{prod['name']}: avg rating = {new_avg:.2f} ({prod['ratingStats']['count']} reviews)")

## Polymorphic Pattern — Multiple shapes

In [ ]:
db.catalog.drop()
db.catalog.insert_many([{"type":"book","name":"MongoDB Guide","price":39.99,"author":"Shannon","pages":514},{"type":"electronics","name":"Headphones","price":149.99,"brand":"Sony","battery":30},{"type":"clothing","name":"Hoodie","price":49.99,"sizes":["S","M","L"],"color":"gray"}])
for item in db.catalog.find({},{"_id":0,"type":1,"name":1,"price":1}):
    print(f"  [{item['type']}] {item['name']}: ${item['price']}")

## Outlier Pattern — Overflow for viral docs

In [ ]:
db.posts.drop()
db.post_overflow.drop()
db.posts.insert_one({"_id":"post-normal","author":"alice","content":"Learning MongoDB","likes":[ObjectId("aaaaaaaaaaaaaaaaaaaaaaaa")],"hasOverflow":False})
db.posts.insert_one({"_id":"post-viral","author":"celebrity","content":"Announcement!","likes":[ObjectId("aaaaaaaaaaaaaaaaaaaaaaaa")],"hasOverflow":True})
db.post_overflow.insert_one({"postId":"post-viral","page":2,"likes":[ObjectId("bbbbbbbbbbbbbbbbbbbbbbbb")]})
viralpost = db.posts.find_one({"_id":"post-viral"})
if viral["hasOverflow"]:
    overflow = list(db.post_overflow.find({"postId":"post-viral"}))
    extra_likes = sum(len(p["likes"]) for p in overflow)
    print(f"Viral post: {viral['content']}, overflow pages={len(overflow)}, extra likes={extra_likes}")

## Subset Pattern — Embed recent, store full elsewhere

In [ ]:
db.products_subset.drop()
db.reviews.drop()
db.products_subset.insert_one({"_id":"prod-monitor","name":"Monitor","price":349.99,"recentReviews":[{"user":"alice","rating":5,"comment":"Excellent!"}],"reviewCount":1,"avgRating":5.0})
db.reviews.insert_one({"productId":"prod-monitor","user":"alice","rating":5,"comment":"Excellent!"})
prod = db.products_subset.find_one({"_id":"prod-monitor"},{"name":1,"recentReviews":1,"reviewCount":1,"_id":0})
print(f"{prod['name']}: {prod['reviewCount']} recent, {len(prod['recentReviews'])} embedded")